# CSE 151B — SFT data prep

Prepare external math datasets into a unified training schema for QLoRA SFT. See `docs/sft/pipeline.md`.

**Sources implemented:** MATH train (`EleutherAI/hendrycks_math`), **NuminaMath-CoT** (`AI-MO/NuminaMath-CoT`), **AGIEval/GaoKao MCQ** (`hails/agieval-*`). Remaining: OpenR1, self-distilled format-fix.

Each source writes:

```text
data/sft_sources/<source>_ready.jsonl
data/sft_sources/<source>_stats.json
data/sft_sources/<source>_rejects.jsonl
```

Run sections in order. Do not mix until every source passes validation.

## 1. Environment

**Colab:** run `%pip`, restart runtime, continue. **Local:** use your venv with `datasets`, `transformers`, `tqdm`.

In [1]:
# # Colab: skip when working locally with an existing venv.
# %pip install -q datasets transformers tqdm

## 2. Setup & paths

Resolves `REPO_ROOT`, optional Drive mount, output dirs, and the same prompt helpers as `dev.ipynb` / `submission.ipynb`.

In [2]:
import hashlib
import json
import os
import random
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

from tqdm.auto import tqdm


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT))

from utils import last_boxed_only_string, remove_boxed

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
PUBLIC_PATH = REPO_ROOT / "data" / "public.jsonl"
PRIVATE_PATH = REPO_ROOT / "data" / "private.jsonl"
SFT_SOURCES_DIR = REPO_ROOT / "data" / "sft_sources"
SFT_SOURCES_DIR.mkdir(parents=True, exist_ok=True)

MAX_TEMPLATE_TOKENS = 7900
MAX_TRACE_CHARS = 12_000
THINKING_OPEN = "<think>"
THINKING_CLOSE = "</think>"
THINKING_TEMPLATE = "explicit_redacted_thinking"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

DRIVE_PROJECT_ROOT: Optional[Path] = None
try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
    print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
except ImportError:
    print("Skip: not Google Colab — using local paths only.")

print(f"REPO_ROOT={REPO_ROOT}")
print(f"SFT_SOURCES_DIR={SFT_SOURCES_DIR}")

Skip: not Google Colab — using local paths only.
REPO_ROOT=/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition
SFT_SOURCES_DIR=/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition/data/sft_sources


/home/andrewyin/UCSD-CSE-Programming-Assignments/CSE151B/151B_SP26_Competition/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Shared helpers

Prompt construction, answer normalization, decontamination, token counting, validation, and JSONL I/O. Reused by every source section.

In [3]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FREEFORM_FINAL_RE = re.compile(r"^\\boxed\{.+\}$")
MCQ_FINAL_RE = re.compile(r"^\\boxed\{[A-J]\}$")
CJK_RE = re.compile(
    r"[\u4e00-\u9fff\u3400-\u4dbf\u3040-\u30ff\uac00-\ud7af]"
)
INLINE_MCQ_LABEL_RE = re.compile(r"\([A-J]\)", re.I)
SEQUENCE_RE = re.compile(
    r"\bsequence\b|\brecurrence\b|\ba_n\b|\ba_\{n\}|\ba_\{\s*n\s*\}", re.I
)
WEAK_TOPICS = {"number_theory", "sequence_recurrence", "geometry"}


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


def normalize_question_for_overlap(text: str) -> str:
    s = text.lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("$", "")
    s = re.sub(r"\\(?:mathrm|mathbf|text|textbf)\{([^}]*)\}", r"\1", s)
    s = re.sub(r"[^a-z0-9\\=+\-*/^_{}().,\[\]]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def load_public_question_keys(public_path: Path) -> set[str]:
    keys: set[str] = set()
    if not public_path.is_file():
        print(f"Warning: {public_path} missing — skipping decontamination.")
        return keys
    with open(public_path) as f:
        for line in f:
            row = json.loads(line)
            keys.add(normalize_question_for_overlap(row["question"]))
    print(f"Loaded {len(keys)} public question keys for decontamination.")
    return keys


def last_non_empty_line(text: str) -> str:
    for line in reversed(text.rstrip().splitlines()):
        if line.strip():
            return line.strip()
    return ""


def extract_boxed_answer(response: str) -> Optional[str]:
    boxed = last_boxed_only_string(response)
    if boxed is None:
        return None
    return remove_boxed(boxed)


def normalize_freeform_response(solution: str) -> tuple[Optional[str], Optional[str]]:
    """Return (response, answer) with a single final \\boxed{...} line, or (None, None)."""
    text = solution.strip()
    if not text:
        return None, None

    answer = extract_boxed_answer(text)
    if answer is None:
        return None, None

    # Drop trailing content after the last boxed answer (rare in MATH).
    boxed = last_boxed_only_string(text)
    assert boxed is not None
    idx = text.rfind(boxed)
    prefix = text[:idx].rstrip()
    final_line = boxed.strip()
    response = f"{prefix}\n{final_line}" if prefix else final_line
    return response, answer.strip()


def validate_final_line(response: str, task_type: str) -> bool:
    line = last_non_empty_line(response)
    if task_type == "mcq":
        return bool(MCQ_FINAL_RE.match(line))
    return bool(FREEFORM_FINAL_RE.match(line))


def contains_cjk(text: str) -> bool:
    return bool(CJK_RE.search(text))


def has_inline_mcq(text: str) -> bool:
    """Detect (A)...(B)...(C) option blocks embedded in problem text."""
    return len(INLINE_MCQ_LABEL_RE.findall(text)) >= 3


def split_reasoning_and_final(response: str) -> tuple[str, str]:
    final_line = last_non_empty_line(response)
    body = response.rstrip()
    if body.endswith(final_line):
        reasoning = body[: -len(final_line)].rstrip()
    else:
        reasoning = body
    return reasoning, final_line


def wrap_thinking_response(response: str) -> str:
    """Reasoning inside <think>; \\boxed{} after the closing tag (D005)."""
    reasoning, final_line = split_reasoning_and_final(response)
    parts = [THINKING_OPEN]
    if reasoning:
        parts.append(reasoning)
    parts.append(THINKING_CLOSE)
    parts.append("")
    parts.append(final_line)
    return "\n".join(parts)


def has_thinking_wrapper(response: str) -> bool:
    return THINKING_OPEN in response and THINKING_CLOSE in response


def render_training_messages(
    question: str,
    options: Optional[list],
    response: str,
) -> list[dict[str, str]]:
    system, user = build_prompt(question, options)
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": response},
    ]


def count_template_tokens(tokenizer, messages: list[dict[str, str]]) -> int:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return len(tokenizer.encode(text, add_special_tokens=False))


def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


@dataclass
class PrepStats:
    source: str
    input_rows: int = 0
    ready_rows: int = 0
    reject_counts: Counter = field(default_factory=Counter)
    topic_counts: Counter = field(default_factory=Counter)
    level_counts: Counter = field(default_factory=Counter)
    trace_char_hist: Counter = field(default_factory=Counter)
    template_token_hist: Counter = field(default_factory=Counter)

    def to_dict(self) -> dict[str, Any]:
        return {
            "source": self.source,
            "input_rows": self.input_rows,
            "ready_rows": self.ready_rows,
            "reject_counts": dict(self.reject_counts),
            "topic_counts": dict(self.topic_counts),
            "level_counts": dict(self.level_counts),
            "trace_char_buckets": dict(self.trace_char_hist),
            "template_token_buckets": dict(self.template_token_hist),
        }


def bucket_value(value: int, edges: list[int]) -> str:
    for edge in edges:
        if value <= edge:
            return f"<={edge}"
    return f">{edges[-1]}"


TRACE_BUCKETS = [500, 1000, 2000, 4000, 8000]
TOKEN_BUCKETS = [512, 1024, 2048, 4096, 7900]


def validate_ready_corpus(
    rows: list[dict[str, Any]],
    source: str,
    *,
    require_thinking_wrapper: bool = False,
) -> None:
    assert rows, f"{source}: no ready rows"
    for i, row in enumerate(rows):
        assert row["source"] == source
        assert row["task_type"] in {"mcq", "freeform"}
        assert validate_final_line(row["response"], row["task_type"])
        if require_thinking_wrapper:
            assert has_thinking_wrapper(row["response"]), f"row {i}: missing thinking wrapper"
        assert row["template_tokens"] <= MAX_TEMPLATE_TOKENS
        assert str(row["answer"]).strip()
        assert not contains_cjk(row["question"])
        assert not contains_cjk(row["response"])
    print(f"Validation passed for {len(rows)} ready rows ({source}).")

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
public_question_keys = load_public_question_keys(PUBLIC_PATH)
private_question_keys = load_public_question_keys(PRIVATE_PATH)
competition_question_keys = public_question_keys | private_question_keys
print(f"Competition decont keys: {len(competition_question_keys)} (public + private)")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded 1126 public question keys for decontamination.
Loaded 943 public question keys for decontamination.
Competition decont keys: 2068 (public + private)


## 4. MATH train (`EleutherAI/hendrycks_math`)

- **Split:** `train` only (7,500 rows across 7 subject configs).
- **Task type:** free-form.
- **Topic tags:** map HF config → topic; keyword-detect `sequence_recurrence`.
- **Weak-topic upweight:** handled at mix time — rows tagged `number_theory`, `geometry`, or `sequence_recurrence` get 3× weight in the final mixer.
- **Response:** MATH `solution` field (short CoT ending in `\boxed{}`).

Filters: must have `\boxed{}`, no overlap with `public.jsonl` questions, final-line regex, ≤ 7900 tokens after chat templating.

In [5]:
from datasets import concatenate_datasets, load_dataset

MATH_SOURCE = "math_train"
MATH_DATASET_ID = "EleutherAI/hendrycks_math"
MATH_CONFIGS = [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "number_theory",
    "prealgebra",
    "precalculus",
]

MATH_CONFIG_TO_TOPIC = {
    "algebra": "algebra",
    "counting_and_probability": "counting_probability",
    "geometry": "geometry",
    "intermediate_algebra": "intermediate_algebra",
    "number_theory": "number_theory",
    "prealgebra": "prealgebra",
    "precalculus": "precalculus",
}

MATH_READY_PATH = SFT_SOURCES_DIR / f"{MATH_SOURCE}_ready.jsonl"
MATH_STATS_PATH = SFT_SOURCES_DIR / f"{MATH_SOURCE}_stats.json"
MATH_REJECTS_PATH = SFT_SOURCES_DIR / f"{MATH_SOURCE}_rejects.jsonl"


def infer_math_topic(config: str, problem: str) -> str:
    if SEQUENCE_RE.search(problem):
        return "sequence_recurrence"
    return MATH_CONFIG_TO_TOPIC[config]


def load_math_train() -> list[dict[str, Any]]:
    parts = []
    for config in MATH_CONFIGS:
        ds = load_dataset(MATH_DATASET_ID, config, split="train")
        ds = ds.add_column("hf_config", [config] * len(ds))
        parts.append(ds)
        print(f"  {config}: {len(ds)} train rows")
    merged = concatenate_datasets(parts)
    print(f"MATH train total: {len(merged)}")
    return merged


def prepare_math_row(
    row: dict[str, Any],
    *,
    row_idx: int,
) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    config = row["hf_config"]
    problem = row["problem"].strip()
    solution = row.get("solution", "")
    source_id = f"{config}:{row_idx}"

    reject_base = {
        "source": MATH_SOURCE,
        "source_id": source_id,
        "hf_config": config,
        "level": row.get("level"),
    }

    if not problem:
        return None, {**reject_base, "reason": "empty_problem"}

    qkey = normalize_question_for_overlap(problem)
    if qkey in public_question_keys:
        return None, {**reject_base, "reason": "public_overlap", "question_preview": problem[:200]}

    response, answer = normalize_freeform_response(solution)
    if response is None or answer is None:
        return None, {**reject_base, "reason": "missing_boxed", "solution_preview": str(solution)[:200]}

    if not validate_final_line(response, "freeform"):
        return None, {**reject_base, "reason": "bad_final_line", "response_tail": response[-120:]}

    topic = infer_math_topic(config, problem)
    messages = render_training_messages(problem, None, response)
    template_tokens = count_template_tokens(tokenizer, messages)
    if template_tokens > MAX_TEMPLATE_TOKENS:
        return None, {
            **reject_base,
            "reason": "too_long_tokens",
            "template_tokens": template_tokens,
        }

    ready = {
        "source": MATH_SOURCE,
        "source_id": source_id,
        "task_type": "freeform",
        "topic": topic,
        "level": row.get("level"),
        "hf_config": config,
        "weak_topic": topic in WEAK_TOPICS,
        "question": problem,
        "options": None,
        "answer": answer,
        "response": response,
        "trace_chars": len(response),
        "template_tokens": template_tokens,
    }
    return ready, None


def run_math_prep() -> tuple[list[dict[str, Any]], list[dict[str, Any]], PrepStats]:
    print("Loading MATH train …")
    raw_rows = load_math_train()

    stats = PrepStats(source=MATH_SOURCE, input_rows=len(raw_rows))
    ready_rows: list[dict[str, Any]] = []
    rejects: list[dict[str, Any]] = []

    config_counters: dict[str, int] = defaultdict(int)
    for row in tqdm(raw_rows, desc="MATH prep"):
        config = row["hf_config"]
        idx = config_counters[config]
        config_counters[config] += 1

        ready, reject = prepare_math_row(row, row_idx=idx)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue

        ready_rows.append(ready)
        stats.topic_counts[ready["topic"]] += 1
        stats.level_counts[ready.get("level") or "unknown"] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(ready["template_tokens"], TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    return ready_rows, rejects, stats

In [6]:
math_ready, math_rejects, math_stats = run_math_prep()

validate_ready_corpus(math_ready, MATH_SOURCE)

write_jsonl(MATH_READY_PATH, math_ready)
write_jsonl(MATH_REJECTS_PATH, math_rejects)

stats_payload = {
    **math_stats.to_dict(),
    "dataset_id": MATH_DATASET_ID,
    "configs": MATH_CONFIGS,
    "ready_path": str(MATH_READY_PATH),
    "rejects_path": str(MATH_REJECTS_PATH),
    "ready_sha256": file_sha256(MATH_READY_PATH),
    "weak_topic_rows": sum(1 for r in math_ready if r["weak_topic"]),
    "mean_trace_chars": round(sum(r["trace_chars"] for r in math_ready) / max(len(math_ready), 1), 1),
    "mean_template_tokens": round(
        sum(r["template_tokens"] for r in math_ready) / max(len(math_ready), 1), 1
    ),
    "seed": RANDOM_SEED,
}

with open(MATH_STATS_PATH, "w") as f:
    json.dump(stats_payload, f, indent=2)

print(json.dumps(stats_payload, indent=2))
print(f"\nWrote {MATH_READY_PATH}")
print(f"Wrote {MATH_REJECTS_PATH} ({len(math_rejects)} rejects)")
print(f"Wrote {MATH_STATS_PATH}")

Loading MATH train …
  algebra: 1744 train rows
  counting_and_probability: 771 train rows
  geometry: 870 train rows
  intermediate_algebra: 1295 train rows
  number_theory: 869 train rows
  prealgebra: 1205 train rows
  precalculus: 746 train rows
MATH train total: 7500


MATH prep: 100%|██████████| 7500/7500 [00:04<00:00, 1706.85it/s]


Validation passed for 7493 ready rows (math_train).
{
  "source": "math_train",
  "input_rows": 7500,
  "ready_rows": 7493,
  "reject_counts": {
    "missing_boxed": 2,
    "bad_final_line": 3,
    "public_overlap": 2
  },
  "topic_counts": {
    "algebra": 1674,
    "sequence_recurrence": 194,
    "counting_probability": 752,
    "geometry": 866,
    "intermediate_algebra": 1227,
    "number_theory": 842,
    "prealgebra": 1201,
    "precalculus": 737
  },
  "level_counts": {
    "Level 5": 2300,
    "Level 3": 1589,
    "Level 4": 1690,
    "Level 1": 564,
    "Level 2": 1348,
    "Level ?": 2
  },
  "trace_char_buckets": {
    "<=500": 4706,
    "<=1000": 1954,
    "<=2000": 715,
    "<=4000": 109,
    "<=8000": 9
  },
  "template_token_buckets": {
    "<=512": 6156,
    "<=1024": 1174,
    "<=2048": 155,
    "<=4096": 8
  },
  "dataset_id": "EleutherAI/hendrycks_math",
  "configs": [
    "algebra",
    "counting_and_probability",
    "geometry",
    "intermediate_algebra",
    "num

### 4.1 Spot-check 20 ready rows

Eyeball format, reasoning quality, and topic tags before moving on.

In [7]:
SPOT_CHECK_N = 20
spot = random.sample(math_ready, k=min(SPOT_CHECK_N, len(math_ready)))

for i, row in enumerate(spot, 1):
    print("=" * 72)
    print(
        f"[{i}] {row['source_id']} | topic={row['topic']} | weak={row['weak_topic']} | "
        f"tokens={row['template_tokens']} | chars={row['trace_chars']}"
    )
    print("Q:", row["question"][:240].replace("\n", " "), "…")
    print("A:", row["answer"][:120])
    print("Final line:", last_non_empty_line(row["response"]))
    print("Tail:", row["response"][-180:].replace("\n", " "))

[1] algebra:913 | topic=algebra | weak=False | tokens=155 | chars=138
Q: What is the sum of the even, positive integers less than 62? …
A: 930
Final line: \boxed{930}
Tail: We are summing up $2+4+6+\cdots+60$. Factoring out a 2 and simplifying, we have $2(1+2+3+\cdots+30)=2\cdot\frac{30\cdot31}{2}= \boxed{930}
[2] algebra:204 | topic=algebra | weak=False | tokens=160 | chars=163
Q: Calculate: $(243)^{\frac35}$ …
A: 27
Final line: \boxed{27}
Tail: We start by finding the prime factorization of 243.  We find $243 = 3^5$, so we have $(243)^{\frac35} = (3^5)^{\frac35} = 3^{5\cdot \frac{3}{5}} = 3^3 = \boxed{27}
[3] prealgebra:531 | topic=prealgebra | weak=False | tokens=335 | chars=390
Q: A circle with center $A$ and radius three inches is tangent at $C$ to a circle with center $B$, as shown. If point $B$ is on the small circle, what is the area of the shaded region? Express your answer in terms of $\pi$.  [asy] filldraw(cir …
A: 27\pi
Final line: \boxed{27\pi}
Tail: maller circle, or six 

### 4.2 Optional: copy artifacts to Drive

Keeps `sft_sources/` recoverable across Colab disconnects.

In [ ]:
# Run section 8 (Drive sync) after all source prep cells have run.
print("Skip: use section 8 after all source prep cells have run.")

## 5. NuminaMath-CoT — Step 2 clean rebuild (`AI-MO/NuminaMath-CoT`)

Pipeline Step 2 per `docs/sft/pipeline.md`. Writes `numina_cot_clean_{ready,stats,rejects}.jsonl`.

- **Split:** `train` (~860k rows), streamed.
- **Task type:** free-form only (inline MCQ rows **rejected**).
- **Filters:** English-only (CJK reject); `trace_chars > 2000` on raw `solution`; drop `gsm8k` / `orca_math`; decontam vs `public.jsonl` + `private.jsonl`; assistant uses explicit `<think>` wrapper (D005); `template_tokens ≤ 7900`; wrapped `trace_chars ≤ 12000`.
- **Cap:** reservoir sample up to `NUMINA_MAX_READY` qualifying rows (default 25k). `NUMINA_MAX_SCAN` limits scan for smoke tests.

Topic tags map the HF `source` field, with keyword overrides for weak topics.

In [7]:
from datasets import concatenate_datasets, load_dataset

NUMINA_SOURCE = "numina_cot_clean"
NUMINA_DATASET_ID = "AI-MO/NuminaMath-CoT"
NUMINA_DROP_SOURCES = {"gsm8k", "orca_math"}
NUMINA_MIN_TRACE_CHARS = 2000
# Reservoir cap on qualifying rows (pre-tokenization). None = keep all (~100k+).
NUMINA_MAX_READY: Optional[int] = 25_000
# Set for local smoke tests only (e.g. 50_000); None = scan full stream.
NUMINA_MAX_SCAN: Optional[int] = None

NUMINA_SOURCE_TO_TOPIC = {
    "olympiads": "olympiads",
    "aops_forum": "aops_forum",
    "amc_aime": "amc_aime",
    "cn_k12": "cn_k12",
    "math": "math",
    "synthetic_math": "synthetic_math",
    "synthetic_amc": "synthetic_amc",
    "gsm8k": "gsm8k",
    "orca_math": "orca_math",
}

NUMBER_THEORY_RE = re.compile(
    r"\b(gcd|lcm|modulo|\bmod\b|remainder|prime|divisor|congruent|coprime)\b",
    re.I,
)
GEOMETRY_RE = re.compile(
    r"\b(triangle|circle|angle|perpendicular|parallel|inradius|circumradius|polygon|circumcircle)\b",
    re.I,
)
BOXED_HINT_RE = re.compile(r"\\boxed\s*\{")

NUMINA_READY_PATH = SFT_SOURCES_DIR / f"{NUMINA_SOURCE}_ready.jsonl"
NUMINA_STATS_PATH = SFT_SOURCES_DIR / f"{NUMINA_SOURCE}_stats.json"
NUMINA_REJECTS_PATH = SFT_SOURCES_DIR / f"{NUMINA_SOURCE}_rejects.jsonl"


def infer_numina_topic(source: str, problem: str) -> str:
    if SEQUENCE_RE.search(problem):
        return "sequence_recurrence"
    if NUMBER_THEORY_RE.search(problem):
        return "number_theory"
    if GEOMETRY_RE.search(problem):
        return "geometry"
    return NUMINA_SOURCE_TO_TOPIC.get(source, source)


def numina_qualifies(row: dict[str, Any], *, source_id: str) -> tuple[bool, Optional[dict[str, Any]]]:
    """Cheap pre-token filters. Returns (ok, reject_dict)."""
    src = row.get("source", "unknown")
    problem = (row.get("problem") or "").strip()
    solution = row.get("solution") or ""

    reject_base = {
        "source": NUMINA_SOURCE,
        "source_id": source_id,
        "hf_source": src,
    }

    if src in NUMINA_DROP_SOURCES:
        return False, {**reject_base, "reason": "dropped_source"}
    if not problem:
        return False, {**reject_base, "reason": "empty_problem"}
    if contains_cjk(problem) or contains_cjk(solution):
        return False, {**reject_base, "reason": "cjk_text", "question_preview": problem[:200]}
    if has_inline_mcq(problem):
        return False, {**reject_base, "reason": "inline_mcq", "question_preview": problem[:200]}
    qkey = normalize_question_for_overlap(problem)
    if qkey in competition_question_keys:
        return False, {
            **reject_base,
            "reason": "competition_overlap",
            "question_preview": problem[:200],
        }
    if len(solution.strip()) <= NUMINA_MIN_TRACE_CHARS:
        return False, {**reject_base, "reason": "short_trace", "trace_chars": len(solution.strip())}
    if not BOXED_HINT_RE.search(solution):
        return False, {**reject_base, "reason": "missing_boxed_hint", "solution_preview": str(solution)[:200]}

    return True, None


def prepare_numina_row(
    row: dict[str, Any],
    *,
    source_id: str,
) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    src = row.get("source", "unknown")
    problem = row["problem"].strip()
    solution = row.get("solution") or ""

    reject_base = {
        "source": NUMINA_SOURCE,
        "source_id": source_id,
        "hf_source": src,
    }

    response, answer = normalize_freeform_response(solution)
    if response is None or answer is None or not str(answer).strip():
        return None, {**reject_base, "reason": "missing_boxed", "solution_preview": str(solution)[:200]}

    if not validate_final_line(response, "freeform"):
        return None, {**reject_base, "reason": "bad_final_line", "response_tail": response[-120:]}

    response = wrap_thinking_response(response)
    if len(response) > MAX_TRACE_CHARS:
        return None, {
            **reject_base,
            "reason": "long_trace",
            "trace_chars": len(response),
        }

    topic = infer_numina_topic(src, problem)
    messages = render_training_messages(problem, None, response)
    template_tokens = count_template_tokens(tokenizer, messages)
    if template_tokens > MAX_TEMPLATE_TOKENS:
        return None, {
            **reject_base,
            "reason": "too_long_tokens",
            "template_tokens": template_tokens,
        }

    ready = {
        "source": NUMINA_SOURCE,
        "source_id": source_id,
        "task_type": "freeform",
        "topic": topic,
        "hf_source": src,
        "weak_topic": topic in WEAK_TOPICS,
        "question": problem,
        "options": None,
        "answer": answer,
        "response": response,
        "trace_chars": len(response),
        "template_tokens": template_tokens,
        "thinking_template": THINKING_TEMPLATE,
    }
    return ready, None


def run_numina_prep() -> tuple[list[dict[str, Any]], list[dict[str, Any]], PrepStats]:
    print(f"Loading {NUMINA_DATASET_ID} (streaming) …")
    stream = load_dataset(NUMINA_DATASET_ID, split="train", streaming=True)

    stats = PrepStats(source=NUMINA_SOURCE)
    rejects: list[dict[str, Any]] = []
    reservoir: list[dict[str, Any]] = []
    qualifying_seen = 0

    bulk_reject_reasons = {
        "dropped_source",
        "short_trace",
        "missing_boxed_hint",
        "cjk_text",
        "inline_mcq",
    }

    for row_idx, row in enumerate(tqdm(stream, desc="Numina scan")):
        stats.input_rows += 1
        source_id = f"{row.get('source', 'unknown')}:{row_idx}"

        ok, reject = numina_qualifies(row, source_id=source_id)
        if not ok:
            reason = reject["reason"]
            stats.reject_counts[reason] += 1
            if reason not in bulk_reject_reasons:
                rejects.append(reject)
            continue

        if NUMINA_MAX_SCAN is not None and stats.input_rows >= NUMINA_MAX_SCAN:
            print(f"Stopping scan at NUMINA_MAX_SCAN={NUMINA_MAX_SCAN}")
            break

        qualifying_seen += 1
        item = dict(row)
        item["_source_id"] = source_id
        cap = NUMINA_MAX_READY
        if cap is None:
            reservoir.append(item)
            continue

        if len(reservoir) < cap:
            reservoir.append(item)
        else:
            j = random.randint(0, qualifying_seen - 1)
            if j < cap:
                reservoir[j] = item

    print(
        f"Qualifying rows: {qualifying_seen}"
        + (
            f" (reservoir cap {NUMINA_MAX_READY} → {len(reservoir)} to tokenize)"
            if NUMINA_MAX_READY
            else ""
        )
    )

    ready_rows: list[dict[str, Any]] = []
    for item in tqdm(reservoir, desc="Numina tokenize"):
        source_id = item["_source_id"]
        row = {k: v for k, v in item.items() if k != "_source_id"}

        ready, reject = prepare_numina_row(row, source_id=source_id)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue

        ready_rows.append(ready)
        stats.topic_counts[ready["topic"]] += 1
        stats.level_counts[ready.get("hf_source") or "unknown"] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(ready["template_tokens"], TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    stats.reject_counts["qualifying_seen"] = qualifying_seen
    return ready_rows, rejects, stats

In [8]:
numina_ready, numina_rejects, numina_stats = run_numina_prep()

validate_ready_corpus(numina_ready, NUMINA_SOURCE, require_thinking_wrapper=True)

write_jsonl(NUMINA_READY_PATH, numina_ready)
write_jsonl(NUMINA_REJECTS_PATH, numina_rejects)

trace_chars = [r["trace_chars"] for r in numina_ready]
template_tokens = [r["template_tokens"] for r in numina_ready]

stats_payload = {
    **numina_stats.to_dict(),
    "dataset_id": NUMINA_DATASET_ID,
    "thinking_template": THINKING_TEMPLATE,
    "ready_path": str(NUMINA_READY_PATH),
    "rejects_path": str(NUMINA_REJECTS_PATH),
    "ready_sha256": file_sha256(NUMINA_READY_PATH) if numina_ready else None,
    "weak_topic_rows": sum(1 for r in numina_ready if r["weak_topic"]),
    "mean_trace_chars": round(sum(trace_chars) / max(len(trace_chars), 1), 1),
    "median_trace_chars": sorted(trace_chars)[len(trace_chars) // 2] if trace_chars else 0,
    "p95_trace_chars": sorted(trace_chars)[int(0.95 * len(trace_chars))] if trace_chars else 0,
    "mean_template_tokens": round(sum(template_tokens) / max(len(template_tokens), 1), 1),
    "median_template_tokens": sorted(template_tokens)[len(template_tokens) // 2]
    if template_tokens
    else 0,
    "p95_template_tokens": sorted(template_tokens)[int(0.95 * len(template_tokens))]
    if template_tokens
    else 0,
    "seed": RANDOM_SEED,
    "numina_max_ready": NUMINA_MAX_READY,
    "numina_max_scan": NUMINA_MAX_SCAN,
}

with open(NUMINA_STATS_PATH, "w") as f:
    json.dump(stats_payload, f, indent=2)

print(json.dumps(stats_payload, indent=2))
print(f"\nWrote {NUMINA_READY_PATH}")
print(f"Wrote {NUMINA_REJECTS_PATH} ({len(numina_rejects)} sampled rejects)")
print(f"Wrote {NUMINA_STATS_PATH}")

Loading AI-MO/NuminaMath-CoT (streaming) …


Numina scan: 859494it [02:18, 6216.67it/s]


Qualifying rows: 99826 (reservoir cap 25000 → 25000 to tokenize)


Numina tokenize: 100%|██████████| 25000/25000 [00:40<00:00, 620.11it/s]


Validation passed for 23089 ready rows (numina_cot_clean).
{
  "source": "numina_cot_clean",
  "input_rows": 859494,
  "ready_rows": 23089,
  "reject_counts": {
    "short_trace": 535762,
    "dropped_source": 160656,
    "inline_mcq": 47723,
    "missing_boxed_hint": 13396,
    "cjk_text": 2124,
    "competition_overlap": 7,
    "missing_boxed": 1620,
    "bad_final_line": 291,
    "qualifying_seen": 99826
  },
  "topic_counts": {
    "aops_forum": 1524,
    "olympiads": 10828,
    "cn_k12": 1981,
    "sequence_recurrence": 1761,
    "geometry": 5632,
    "number_theory": 1203,
    "amc_aime": 36,
    "math": 59,
    "synthetic_amc": 44,
    "synthetic_math": 21
  },
  "level_counts": {
    "aops_forum": 2571,
    "olympiads": 17181,
    "cn_k12": 3084,
    "amc_aime": 66,
    "math": 102,
    "synthetic_amc": 54,
    "synthetic_math": 31
  },
  "trace_char_buckets": {
    "<=4000": 22430,
    "<=8000": 282,
    "<=2000": 327,
    "<=500": 10,
    "<=1000": 37,
    ">8000": 3
  },
  "

### 5.1 Run clean prep + spot-check

Run the cell below first (streams Numina — long on first full run). Then spot-check 20 rows.

In [10]:
SPOT_CHECK_N = 20
spot_numina = random.sample(numina_ready, k=min(SPOT_CHECK_N, len(numina_ready)))

for i, row in enumerate(spot_numina, 1):
    print("=" * 72)
    print(
        f"[{i}] {row['source_id']} | hf={row.get('hf_source')} | topic={row['topic']} | "
        f"weak={row['weak_topic']} | tokens={row['template_tokens']} | chars={row['trace_chars']}"
    )
    print("Q:", row["question"][:240].replace("\n", " "), "…")
    print("A:", row["answer"][:120])
    print("Final line:", last_non_empty_line(row["response"]))
    print("Thinking wrapper:", has_thinking_wrapper(row["response"]))
    print("Tail:", row["response"][-220:].replace("\n", " "))

[1] olympiads:89533 | hf=olympiads | topic=olympiads | weak=False | tokens=966 | chars=2055
Q: Determine the positive integer solutions of the following equation:  $$ xy z + 2x + 3y + 6z = xy + 2xz + 3yz $$ …
A: (x, y, z) = (4, 3, 1)
Final line: \boxed{(x, y, z) = (4, 3, 1)}
Thinking wrapper: True
Tail:     \]  7. **Consider all valid positive integer solutions:**        Upon validation and correction of all combinations, we get:     \( (x, y, z) = (4, 3, 1) \)  ### Conclusion  \[ </think>  \boxed{(x, y, z) = (4, 3, 1)}
[2] cn_k12:342873 | hf=cn_k12 | topic=cn_k12 | weak=False | tokens=1169 | chars=2670
Q: Given that $\sin(3\pi - \alpha) = 2\sin\left(\frac{\pi}{2} + \alpha\right)$, find the value of $\frac{\sin^3(\pi - \alpha) - \sin\left(\frac{\pi}{2} - \alpha\right)}{3\cos\left(\frac{\pi}{2} + \alpha\right) + 2\cos(\pi + \alpha)}$. …
A: \frac{8 - 5\sqrt{5}}{40\sqrt{5}}
Final line: \boxed{\frac{8 - 5\sqrt{5}}{40\sqrt{5}}}
Thinking wrapper: True
Tail: } - \frac{\sqrt{5}}{\sqrt{5}}}{8} 

### 5.2 Corpus audit

Re-audit **all** ready rows (in-memory `numina_ready` or `numina_cot_clean_ready.jsonl` on disk). Run after §5.1. Use output to decide whether to rebuild filters, raise `NUMINA_MAX_READY`, or downsample at Step 5.

In [11]:
import statistics

BOXED_IN_THINKING_RE = re.compile(r"\\boxed\s*\{")
DANGLING_CLOSE_RE = re.compile(r"(Conclusion:\s*\\\[|\\\[\s*)$")


def _numina_thinking_body(response: str) -> str:
    if THINKING_CLOSE not in response:
        return ""
    return response.split(THINKING_CLOSE, 1)[0].split(THINKING_OPEN, 1)[-1]


def _numina_final_line(response: str) -> str:
    after = response.split(THINKING_CLOSE, 1)[-1].strip()
    return after.splitlines()[-1] if after else ""


def audit_numina_corpus(rows: list[dict[str, Any]]) -> dict[str, Any]:
    n = len(rows)
    if n == 0:
        return {"n": 0}

    trace_chars = [r["trace_chars"] for r in rows]
    template_tokens = [r["template_tokens"] for r in rows]
    hf_sources: Counter = Counter()
    topics: Counter = Counter()

    issues: Counter = Counter()
    letter_final_rows: list[dict[str, Any]] = []
    short_rows: list[dict[str, Any]] = []
    multi_boxed_rows: list[dict[str, Any]] = []

    for r in rows:
        q, resp = r["question"], r["response"]
        hf_sources[r.get("hf_source", "?")] += 1
        topics[r["topic"]] += 1

        if not has_thinking_wrapper(resp):
            issues["missing_thinking_wrapper"] += 1
        fl = _numina_final_line(resp)
        if not validate_final_line(resp, "freeform"):
            issues["bad_final_regex"] += 1
        if contains_cjk(q) or contains_cjk(resp):
            issues["cjk_leak"] += 1
        if has_inline_mcq(q):
            issues["inline_mcq_leak"] += 1
        if normalize_question_for_overlap(q) in competition_question_keys:
            issues["competition_overlap"] += 1
        if r["template_tokens"] > MAX_TEMPLATE_TOKENS:
            issues["over_token_cap"] += 1
        if r["trace_chars"] > MAX_TRACE_CHARS:
            issues["over_char_cap"] += 1

        tb = _numina_thinking_body(resp)
        if BOXED_IN_THINKING_RE.search(tb):
            issues["extra_boxed_in_thinking"] += 1
            if len(multi_boxed_rows) < 5:
                multi_boxed_rows.append(r)
        if DANGLING_CLOSE_RE.search(tb.rstrip()[-80:]):
            issues["dangling_latex_before_close"] += 1
        if len(tb.strip()) < 100:
            issues["thin_thinking"] += 1
        if r["trace_chars"] < NUMINA_MIN_TRACE_CHARS:
            issues["post_wrap_under_min_trace"] += 1
            short_rows.append(r)
        if MCQ_FINAL_RE.match(fl):
            issues["letter_final_mcq_style"] += 1
            if len(letter_final_rows) < 5:
                letter_final_rows.append(r)

    def pct(x: int) -> float:
        return 100.0 * x / n

    return {
        "n": n,
        "trace_chars": trace_chars,
        "template_tokens": template_tokens,
        "hf_sources": hf_sources,
        "topics": topics,
        "issues": issues,
        "letter_final_rows": letter_final_rows,
        "short_rows": short_rows,
        "multi_boxed_rows": multi_boxed_rows,
        "weak_topic_rows": sum(1 for r in rows if r.get("weak_topic")),
        "pct": pct,
    }


# Load corpus: prefer in-memory from §5.1, else disk
try:
    _audit_rows = numina_ready
    _audit_source = "in-memory numina_ready"
except NameError:
    if NUMINA_READY_PATH.is_file():
        _audit_rows = read_jsonl(NUMINA_READY_PATH)
        _audit_source = str(NUMINA_READY_PATH)
    else:
        _audit_rows = []
        _audit_source = "(none — run §5.1 first)"
if not _audit_rows and NUMINA_READY_PATH.is_file():
    _audit_rows = read_jsonl(NUMINA_READY_PATH)
    _audit_source = str(NUMINA_READY_PATH)

report = audit_numina_corpus(_audit_rows)
n = report["n"]
pct = report["pct"]

print(f"Audit source: {_audit_source}")
print(f"Rows: {n:,}")
if n == 0:
    raise SystemExit("No rows to audit.")

tc = report["trace_chars"]
tt = report["template_tokens"]
print(
    f"trace_chars: mean={statistics.mean(tc):.0f} median={statistics.median(tc):.0f} "
    f"min={min(tc)} max={max(tc)} p5={sorted(tc)[max(0, int(0.05 * n) - 1)]} "
    f"p95={sorted(tc)[min(n - 1, int(0.95 * n))]}"
)
print(
    f"template_tokens: mean={statistics.mean(tt):.0f} median={statistics.median(tt):.0f} "
    f"max={max(tt)}"
)
print(f"weak_topic: {report['weak_topic_rows']:,} ({pct(report['weak_topic_rows']):.1f}%)")
print()

if NUMINA_STATS_PATH.is_file():
    saved = json.loads(NUMINA_STATS_PATH.read_text())
    inp = saved.get("input_rows", 0)
    qual = saved.get("reject_counts", {}).get("qualifying_seen", 0)
    cap = saved.get("numina_max_ready")
    print("Funnel (from numina_cot_clean_stats.json)")
    print(f"  scanned: {inp:,}")
    print(f"  qualifying (pre-token): {qual:,}" + (f" ({100 * qual / inp:.1f}% of scan)" if inp else ""))
    print(f"  ready: {n:,}" + (f" ({100 * n / qual:.1f}% of qualifying)" if qual else ""))
    if cap is not None and qual > cap:
        est = int(qual * n / min(cap, qual))
        print(
            f"  NUMINA_MAX_READY={cap:,} capped tokenization — only {min(cap, qual):,} rows tokenized."
        )
        print(f"  Estimated ready if all qualifying tokenized (~{n / min(cap, qual):.1%} pass): ~{est:,}")
    print("  Scan rejects:")
    for reason, cnt in sorted(
        ((k, v) for k, v in saved.get("reject_counts", {}).items() if k != "qualifying_seen"),
        key=lambda x: -x[1],
    )[:8]:
        print(f"    {reason}: {cnt:,}")
    print()

print("HF source (top 5):")
for k, v in report["hf_sources"].most_common(5):
    print(f"  {k}: {v:,} ({pct(v):.1f}%)")
print()
print("Quality issues (re-audit; 0 = pass):")
for key in sorted(report["issues"]):
    c = report["issues"][key]
    print(f"  {key}: {c:,} ({pct(c):.1f}%)")
print()

# Step 4 gate: >2 hard failures → rebuild
hard = (
    report["issues"]["missing_thinking_wrapper"]
    + report["issues"]["bad_final_regex"]
    + report["issues"]["cjk_leak"]
    + report["issues"]["inline_mcq_leak"]
    + report["issues"]["competition_overlap"]
    + sum(1 for r in report["short_rows"] if r["trace_chars"] < 500)
)
print(f"Step 4 hard-failure count (heuristic): {hard}  (pipeline: rebuild if > 2)")
if report["issues"]["letter_final_mcq_style"]:
    print(
        f"  letter_final_mcq_style: {report['issues']['letter_final_mcq_style']:,} — "
        "consider rejecting \\boxed{{A-J}} free-form finals"
    )
print()

if report["letter_final_rows"]:
    print("Sample letter-final rows:")
    for r in report["letter_final_rows"][:3]:
        print(f"  {r['source_id']} | final={_numina_final_line(r['response'])}")
        print(f"    Q: {r['question'][:140].replace(chr(10), ' ')}…")
    print()

if report["short_rows"]:
    print("Shortest rows (post-wrap trace_chars < NUMINA_MIN_TRACE_CHARS):")
    for r in sorted(report["short_rows"], key=lambda x: x["trace_chars"])[:5]:
        print(
            f"  {r['trace_chars']} chars | {r['template_tokens']} tok | {r['source_id']} | "
            f"final={_numina_final_line(r['response'])[:50]}"
        )
    print()

if report["multi_boxed_rows"]:
    print("Sample extra-\\boxed in thinking:")
    for r in report["multi_boxed_rows"][:2]:
        tb = _numina_thinking_body(r["response"])
        print(
            f"  {r['source_id']} | boxes_in_think={len(BOXED_IN_THINKING_RE.findall(tb))} | "
            f"final={_numina_final_line(r['response'])[:60]}"
        )

Audit source: in-memory numina_ready
Rows: 23,089
trace_chars: mean=2507 median=2395 min=148 max=9160 p5=2037 p95=3376
template_tokens: mean=1102 median=1066 max=3731
weak_topic: 8,596 (37.2%)

Funnel (from numina_cot_clean_stats.json)
  scanned: 859,494
  qualifying (pre-token): 99,826 (11.6% of scan)
  ready: 23,089 (23.1% of qualifying)
  NUMINA_MAX_READY=25,000 capped tokenization — only 25,000 rows tokenized.
  Estimated ready if all qualifying tokenized (~92.4% pass): ~92,195
  Scan rejects:
    short_trace: 535,762
    dropped_source: 160,656
    inline_mcq: 47,723
    missing_boxed_hint: 13,396
    cjk_text: 2,124
    missing_boxed: 1,620
    bad_final_line: 291
    competition_overlap: 7

HF source (top 5):
  olympiads: 17,181 (74.4%)
  cn_k12: 3,084 (13.4%)
  aops_forum: 2,571 (11.1%)
  math: 102 (0.4%)
  amc_aime: 66 (0.3%)

Quality issues (re-audit; 0 = pass):
  dangling_latex_before_close: 14,582 (63.2%)
  extra_boxed_in_thinking: 3,722 (16.1%)
  letter_final_mcq_style: 41

## 6. AGIEval-Math + GaoKao-MCQ

MCQ format coverage for SFT. Combines three HuggingFace `hails/agieval-*` subsets (825 rows total):

| Subset | Dataset | Rows | Original options |
|--------|---------|------|------------------|
| GaoKao | `hails/agieval-gaokao-mathqa` | 351 | 4 |
| SAT math | `hails/agieval-sat-math` | 220 | 4 |
| AQuA-RAT | `hails/agieval-aqua-rat` | 254 | 5 |

**Note:** `hails/agieval-math` is free-form QA (no choices) and is excluded here.

- **Task type:** MCQ only.
- **10-option expansion:** sample wrong choices from sibling problems in the same subset, shuffle, remap gold letter.
- **Response:** synthetic short CoT ending in `\boxed{<letter>}` (sources ship without reasoning traces).
- **Filters:** decontamination vs `public.jsonl`, final-line regex, ≤ 7900 tokens after templating.

In [ ]:
AGIEVAL_MCQ_SOURCE = "agieval_mcq"
TARGET_MCQ_OPTIONS = 10
CHOICE_LABEL_RE = re.compile(r"^\([A-J]\)\s*")

AGIEVAL_MCQ_DATASETS = [
    {
        "dataset_id": "hails/agieval-gaokao-mathqa",
        "hf_subset": "gaokao",
        "split": "test",
        "topic": "gaokao",
    },
    {
        "dataset_id": "hails/agieval-sat-math",
        "hf_subset": "sat_math",
        "split": "test",
        "topic": "sat_math",
    },
    {
        "dataset_id": "hails/agieval-aqua-rat",
        "hf_subset": "aqua_rat",
        "split": "test",
        "topic": "aqua_rat",
    },
]

AGIEVAL_MCQ_READY_PATH = SFT_SOURCES_DIR / f"{AGIEVAL_MCQ_SOURCE}_ready.jsonl"
AGIEVAL_MCQ_STATS_PATH = SFT_SOURCES_DIR / f"{AGIEVAL_MCQ_SOURCE}_stats.json"
AGIEVAL_MCQ_REJECTS_PATH = SFT_SOURCES_DIR / f"{AGIEVAL_MCQ_SOURCE}_rejects.jsonl"


def strip_choice_label(text: str) -> str:
    return CHOICE_LABEL_RE.sub("", text.strip()).strip()


def parse_hails_query_question(query: str) -> str:
    q = query.strip()
    if q.startswith("Q:"):
        q = q[2:].strip()
        if " Answer Choices:" in q:
            q = q.split(" Answer Choices:")[0].strip()
        elif "\nA:" in q:
            q = q.split("\nA:")[0].strip()
    elif q.startswith("问题："):
        q = q.split("问题：", 1)[1]
        for sep in ["\n 选项：", "\n选项：", "选项："]:
            if sep in q:
                q = q.split(sep)[0].strip()
                break
        q = q.split("\n答案：")[0].strip()
    return q.strip()


def gold_to_letter(gold: list, n_choices: int) -> Optional[str]:
    if not gold:
        return None
    idx = int(gold[0])
    if idx < 0 or idx >= n_choices:
        return None
    return chr(65 + idx)


def synthesize_mcq_response(correct_letter: str) -> str:
    return (
        "I'll solve this step by step and compare the answer choices.\n\n"
        "Working through the problem, I eliminate inconsistent options "
        "and verify the remaining candidate against the constraints.\n\n"
        f"The correct choice is {correct_letter}.\n"
        f"\\boxed{{{correct_letter}}}"
    )


def expand_mcq_options(
    options: list[str],
    correct_idx: int,
    distractor_pool: list[str],
    *,
    target: int,
    rng: random.Random,
) -> tuple[Optional[list[str]], Optional[str], Optional[str]]:
    if correct_idx < 0 or correct_idx >= len(options):
        return None, None, "bad_correct_idx"

    needed = max(0, target - len(options))
    combined = list(options)
    if needed:
        existing = set(options)
        distractors: list[str] = []
        candidates = [d for d in distractor_pool if d not in existing]
        rng.shuffle(candidates)
        for candidate in candidates:
            if len(distractors) >= needed:
                break
            if candidate not in existing:
                distractors.append(candidate)
                existing.add(candidate)
        if len(options) + len(distractors) < target:
            return None, None, "insufficient_distractors"
        combined = options + distractors

    perm = list(range(len(combined)))
    rng.shuffle(perm)
    new_options = [combined[i] for i in perm]
    new_correct_idx = perm.index(correct_idx)
    new_letter = chr(65 + new_correct_idx)
    return new_options, new_letter, None


def infer_agieval_topic(hf_subset: str, problem: str) -> str:
    if SEQUENCE_RE.search(problem):
        return "sequence_recurrence"
    return hf_subset


def load_agieval_mcq_raw() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for spec in AGIEVAL_MCQ_DATASETS:
        ds = load_dataset(spec["dataset_id"], split=spec["split"])
        print(f"  {spec['hf_subset']}: {len(ds)} rows from {spec['dataset_id']}")
        for idx, row in enumerate(ds):
            rows.append(
                {
                    "hf_subset": spec["hf_subset"],
                    "topic_default": spec["topic"],
                    "row_idx": idx,
                    "query": row["query"],
                    "choices": row.get("choices") or [],
                    "gold": row.get("gold") or [],
                }
            )
    print(f"AGIEval/GaoKao MCQ total: {len(rows)}")
    return rows


def normalize_agieval_row(row: dict[str, Any]) -> tuple[Optional[dict[str, Any]], Optional[dict[str, Any]]]:
    hf_subset = row["hf_subset"]
    row_idx = row["row_idx"]
    source_id = f"{hf_subset}:{row_idx}"
    reject_base = {
        "source": AGIEVAL_MCQ_SOURCE,
        "source_id": source_id,
        "hf_subset": hf_subset,
    }

    choices_raw = row.get("choices") or []
    if not choices_raw:
        return None, {**reject_base, "reason": "missing_choices"}

    options = [strip_choice_label(c) for c in choices_raw]
    if not all(options):
        return None, {**reject_base, "reason": "empty_choice_text"}

    question = parse_hails_query_question(row["query"])
    if not question:
        return None, {**reject_base, "reason": "empty_problem"}

    letter = gold_to_letter(row.get("gold") or [], len(options))
    if letter is None:
        return None, {**reject_base, "reason": "bad_gold_index", "gold": row.get("gold")}

    correct_idx = ord(letter) - 65
    return {
        "source_id": source_id,
        "hf_subset": hf_subset,
        "question": question,
        "options": options,
        "correct_idx": correct_idx,
        "correct_letter": letter,
        "original_n_options": len(options),
    }, None


def run_agieval_mcq_prep() -> tuple[list[dict[str, Any]], list[dict[str, Any]], PrepStats]:
    print("Loading AGIEval/GaoKao MCQ sources …")
    raw_rows = load_agieval_mcq_raw()
    stats = PrepStats(source=AGIEVAL_MCQ_SOURCE, input_rows=len(raw_rows))
    rejects: list[dict[str, Any]] = []

    normalized: list[dict[str, Any]] = []
    by_subset: dict[str, list[dict[str, Any]]] = defaultdict(list)

    for row in raw_rows:
        norm, reject = normalize_agieval_row(row)
        if reject:
            rejects.append(reject)
            stats.reject_counts[reject["reason"]] += 1
            continue
        normalized.append(norm)
        by_subset[norm["hf_subset"]].append(norm)

    distractor_pools = {
        subset: [
            opt
            for item in items
            for i, opt in enumerate(item["options"])
            if i != item["correct_idx"]
        ]
        for subset, items in by_subset.items()
    }

    ready_rows: list[dict[str, Any]] = []
    for norm in tqdm(normalized, desc="AGIEval MCQ prep"):
        reject_base = {
            "source": AGIEVAL_MCQ_SOURCE,
            "source_id": norm["source_id"],
            "hf_subset": norm["hf_subset"],
        }

        qkey = normalize_question_for_overlap(norm["question"])
        if qkey in public_question_keys:
            reject = {
                **reject_base,
                "reason": "public_overlap",
                "question_preview": norm["question"][:200],
            }
            rejects.append(reject)
            stats.reject_counts["public_overlap"] += 1
            continue

        rng = random.Random(RANDOM_SEED + hash(norm["source_id"]))
        expanded_options, answer, err = expand_mcq_options(
            norm["options"],
            norm["correct_idx"],
            distractor_pools[norm["hf_subset"]],
            target=TARGET_MCQ_OPTIONS,
            rng=rng,
        )
        if err:
            reject = {
                **reject_base,
                "reason": err,
                "original_n_options": norm["original_n_options"],
            }
            rejects.append(reject)
            stats.reject_counts[err] += 1
            continue

        response = synthesize_mcq_response(answer)
        if not validate_final_line(response, "mcq"):
            reject = {
                **reject_base,
                "reason": "bad_final_line",
                "response_tail": response[-120:],
            }
            rejects.append(reject)
            stats.reject_counts["bad_final_line"] += 1
            continue

        topic = infer_agieval_topic(norm["hf_subset"], norm["question"])
        messages = render_training_messages(norm["question"], expanded_options, response)
        template_tokens = count_template_tokens(tokenizer, messages)
        if template_tokens > MAX_TEMPLATE_TOKENS:
            reject = {
                **reject_base,
                "reason": "too_long_tokens",
                "template_tokens": template_tokens,
            }
            rejects.append(reject)
            stats.reject_counts["too_long_tokens"] += 1
            continue

        ready = {
            "source": AGIEVAL_MCQ_SOURCE,
            "source_id": norm["source_id"],
            "task_type": "mcq",
            "topic": topic,
            "hf_subset": norm["hf_subset"],
            "weak_topic": topic in WEAK_TOPICS,
            "question": norm["question"],
            "options": expanded_options,
            "answer": answer,
            "response": response,
            "trace_chars": len(response),
            "template_tokens": template_tokens,
            "original_n_options": norm["original_n_options"],
            "expanded_to": TARGET_MCQ_OPTIONS,
        }
        ready_rows.append(ready)
        stats.topic_counts[topic] += 1
        stats.level_counts[norm["hf_subset"]] += 1
        stats.trace_char_hist[bucket_value(ready["trace_chars"], TRACE_BUCKETS)] += 1
        stats.template_token_hist[bucket_value(template_tokens, TOKEN_BUCKETS)] += 1

    stats.ready_rows = len(ready_rows)
    return ready_rows, rejects, stats

In [ ]:
agieval_mcq_ready, agieval_mcq_rejects, agieval_mcq_stats = run_agieval_mcq_prep()

validate_ready_corpus(agieval_mcq_ready, AGIEVAL_MCQ_SOURCE)

write_jsonl(AGIEVAL_MCQ_READY_PATH, agieval_mcq_ready)
write_jsonl(AGIEVAL_MCQ_REJECTS_PATH, agieval_mcq_rejects)

agieval_stats_payload = {
    **agieval_mcq_stats.to_dict(),
    "datasets": AGIEVAL_MCQ_DATASETS,
    "target_mcq_options": TARGET_MCQ_OPTIONS,
    "ready_path": str(AGIEVAL_MCQ_READY_PATH),
    "rejects_path": str(AGIEVAL_MCQ_REJECTS_PATH),
    "ready_sha256": file_sha256(AGIEVAL_MCQ_READY_PATH),
    "mean_trace_chars": round(
        sum(r["trace_chars"] for r in agieval_mcq_ready) / max(len(agieval_mcq_ready), 1), 1
    ),
    "mean_template_tokens": round(
        sum(r["template_tokens"] for r in agieval_mcq_ready) / max(len(agieval_mcq_ready), 1), 1
    ),
    "seed": RANDOM_SEED,
}

with open(AGIEVAL_MCQ_STATS_PATH, "w") as f:
    json.dump(agieval_stats_payload, f, indent=2)

print(json.dumps(agieval_stats_payload, indent=2))
print(f"\nWrote {AGIEVAL_MCQ_READY_PATH}")
print(f"Wrote {AGIEVAL_MCQ_REJECTS_PATH} ({len(agieval_mcq_rejects)} rejects)")
print(f"Wrote {AGIEVAL_MCQ_STATS_PATH}")


### 6.1 Spot-check 20 ready rows

In [ ]:
spot_agieval = random.sample(agieval_mcq_ready, k=min(SPOT_CHECK_N, len(agieval_mcq_ready)))

for i, row in enumerate(spot_agieval, 1):
    print("=" * 72)
    print(
        f"[{i}] {row['source_id']} | subset={row.get('hf_subset')} | topic={row['topic']} | "
        f"n_opts={len(row['options'])} (from {row['original_n_options']}) | "
        f"tokens={row['template_tokens']} | chars={row['trace_chars']}"
    )
    print("Q:", row["question"][:240].replace("\n", " "), "…")
    print("A:", row["answer"])
    print("Options[:3]:", row["options"][:3])
    print("Final line:", last_non_empty_line(row["response"]))

## 7. Step 5 — final corpus (Numina-only first run)

Build `data/sft_corpus.jsonl` from `numina_cot_clean_ready.jsonl`:

- Drop `trace_chars < 500` (audit: 10 rows)
- Drop `letter_final_mcq_style` (`\\boxed{A-J}` on free-form rows)
- Weighted downsample: weak topics get **3×** weight
- Default target **15,000** rows (pipeline A100 40 GB band 12k–18k)

Local: `python scripts/build_sft_corpus.py` (flags: `--target`, `--seed`, `--keep-letter-final`).

## 8. Future sources (not yet implemented)

1. **OpenR1-Math distill** — after license check; remap thinking tags to Qwen3 schema.
2. **Baseline-self-distilled format-fix** — external problems only, rewrite final line.

When adding sources, extend the mixer to combine multiple `*_ready.jsonl` files.

In [ ]:
import subprocess
import sys

_corpus_script = REPO_ROOT / "scripts" / "build_sft_corpus.py"
subprocess.run([sys.executable, str(_corpus_script)], check=True, cwd=REPO_ROOT)

_manifest = json.loads((REPO_ROOT / "data" / "sft_corpus_manifest.json").read_text())
print(f"Corpus rows: {_manifest['final_row_count']:,}")
print(f"Corpus sha256: {_manifest['corpus_sha256']}")

## 9. Optional: copy artifacts to Drive

Keeps `sft_sources/` recoverable across Colab disconnects.


In [ ]:
import shutil

if DRIVE_PROJECT_ROOT is not None:
    drive_data = DRIVE_PROJECT_ROOT / "data"
    drive_sources = drive_data / "sft_sources"
    drive_sources.mkdir(parents=True, exist_ok=True)
    paths = [
        MATH_READY_PATH,
        MATH_STATS_PATH,
        MATH_REJECTS_PATH,
        NUMINA_READY_PATH,
        NUMINA_STATS_PATH,
        NUMINA_REJECTS_PATH,
        AGIEVAL_MCQ_READY_PATH,
        AGIEVAL_MCQ_STATS_PATH,
        AGIEVAL_MCQ_REJECTS_PATH,
        REPO_ROOT / "data" / "sft_corpus.jsonl",
        REPO_ROOT / "data" / "sft_corpus_manifest.json",
    ]
    for path in paths:
        if not path.is_file():
            continue
        dest = drive_data / path.name if path.name.startswith("sft_corpus") else drive_sources / path.name
        shutil.copy2(path, dest)
    print(f"Copied SFT artifacts under {drive_data}")
else:
    print("Skip: no Drive mount.")